# Session 2 
### Iteration 1 - Chunking and Embedding

For this class exercise I downloaded the local governmental regulations from my local municipality (boro).  There are 3 PDF files which hold all administrative, general, and franchise legislations for my boro.  These documents will be the source of my RAG database. They are stored in the "data" folder in this project. 

#### New concepts for this iteration:
 - Chunking and embedding documents into a vector database  

#### We will use ChromaDB and the default embedding model



In [1]:
import re
import chromadb
import pypdf
from pathlib import Path

# This regex pattern is used to extract sections from the PDF files.
SECTION_PATTERN = re.compile(
    r'(?=(?:Chapter\s+[A-Z]?\d+|ARTICLE\s+[IVXLC]+|§\s*[A-Z]?\d+[-\w]*\.))',
    re.MULTILINE
)
# This is the number of characters to overlap between chunks.
OVERLAP_CHARS = 200

# This function extracts sections from a PDF file.
# It splits the PDF file into chunks based on the regex pattern.
# It then creates a dictionary for each chunk with the following keys:
# - text: the text of the chunk
# - source: the source of the chunk (the name of the PDF file)
# - chunk_index: the index of the chunk
# - heading: the heading of the chunk
def extract_sections(pdf_path: Path) -> list[dict]:
    reader = pypdf.PdfReader(pdf_path)
    full_text = "\n".join(page.extract_text() or "" for page in reader.pages)
    raw_chunks = SECTION_PATTERN.split(full_text)
    raw_chunks = [c.strip() for c in raw_chunks if c.strip()]

    sections = []
    for i, chunk in enumerate(raw_chunks):
        overlap = raw_chunks[i - 1][-OVERLAP_CHARS:] if i > 0 else ""
        heading = chunk.splitlines()[0].strip()
        sections.append({
            "text": overlap + chunk,
            "source": pdf_path.name,
            "chunk_index": i,
            "heading": heading,
        })
    return sections

In [ ]:
client = chromadb.EphemeralClient()
collection = client.get_or_create_collection("boro_regulations")

data_dir = Path("data/baldwin")
all_sections = []
for pdf_path in sorted(data_dir.glob("*.pdf")):
    all_sections.extend(extract_sections(pdf_path))

collection.add(
    documents=[s["text"] for s in all_sections],
    metadatas=[{"source": s["source"], "chunk_index": s["chunk_index"], "heading": s["heading"]} for s in all_sections],
    ids=[f"{s['source']}_chunk_{s['chunk_index']}" for s in all_sections],
)
print(f"Added {len(all_sections)} chunks from {len(list(data_dir.glob('*.pdf')))} PDFs")

Added 2061 chunks from 3 PDFs


In [3]:
def retrieve(query: str, n_results: int = 5) -> list[dict]:
    results = collection.query(query_texts=[query], n_results=n_results)
    return [
        {
            "text": text,
            "metadata": meta,
            "score": 1 - distance,
        }
        for text, meta, distance in zip(
            results["documents"][0],
            results["metadatas"][0],
            results["distances"][0],
        )
    ]

In [4]:
results = retrieve("do i have to register with the boro if i want to install an alarm?")

for r in results:
    print(f"Score: {r['score']:.3f} | {r['metadata']['source']} — {r['metadata']['heading']}")
    print(r['text'][:300])
    print("---")

Score: 0.142 | BaldwinBoro-GeneralLegislation.pdf — §  62-6. Duties of alarm user.
assified 
as use of a nonregistered alarm system, and citations and penalties shall be assessed without waiver. A 
twenty-five-dollar late fee can be assessed if the renewal is more than 30 days late.§  62-6. Duties of alarm user. 
E. Any false statement of a material fact made by an applicant for t
---
Score: 0.089 | BaldwinBoro-GeneralLegislation.pdf — §  62-7. Duties of alarm company; permit required; fee.
function in the alarm system to be more false alarm resistant or provide additional user 
Borough of Baldwin, PA
§  62-3 ALARM SYSTEMS §  62-6
Downloaded from https://ecode360.com/BA0968 on 2026-06-17§  62-7. Duties of alarm company; permit required; fee. 
training as appropriate. See Appendix A for
---
Score: 0.079 | BaldwinBoro-GeneralLegislation.pdf — §  62-7. Duties of alarm company;
§  62-6. Duties of alarm user.§  62-7. Duties of alarm company; 
permit required; fee.
---
Score: 0.069 | Baldwin